<a id="inizio"></a>
# 10 · Grafici con Plotly

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Base &nbsp;·&nbsp; ⏱ ~40 min**

Il capo non legge i DataFrame, legge i grafici. Plotly Express ne fa uno per riga di codice, interattivo, e lo salva in un file HTML che si apre senza Python.

**In questo notebook impariamo a**

- fare un grafico a linee, un istogramma e uno scatter con Plotly Express
- confrontare più serie e navigarle con slider e selettore di intervallo
- esportare un grafico in HTML da mandare a chi non ha Python

Prima di questo: [09 · pandas: le date](09_Pandas_date.ipynb) &nbsp;·&nbsp; Dati: `../Dati/prezzi_zonali_2025_settimana.csv`, `../Dati/TexasTurbine.csv`, `../Dati/load_total_north_hourly_2024.xlsx`

</div>

<a id="indice"></a>
**Indice**

- [1. I quattro pezzi di un grafico](#sez-1)
- [2. Linee con `px.line`](#sez-2)
- [3. Istogrammi e punti](#sez-3)
- [4. Più serie e navigazione](#sez-4)
- [5. Esportare](#sez-5)
- [6. Esercizi](#sez-6)
    - [Esercizio 10.1 · Il grafico per il capo](#es-10-1)
    - [Esercizio 10.1 bis · Zone a confronto](#es-10-1-bis)
    - [Esercizio 10.2 · Vento e potenza](#es-10-2)
    - [Esercizio 10.2 bis · Temperatura e potenza](#es-10-2-bis)

<a id="sez-1"></a>
## 1. I quattro pezzi di un grafico

Un grafico Plotly è fatto di quattro pezzi.

- I dati: un DataFrame.
- Il mapping: quale colonna va sulla x, quale sulla y, quale decide il colore.
- Le tracce: le linee o i punti che ne risultano.
- Il layout: titolo, assi, dimensioni.

Plotly Express, `px`, costruisce i primi tre con una chiamata sola. Ripartiamo dai prezzi zonali.

In [ ]:
import pandas as pd
import plotly.express as px

prezzi = pd.read_csv("../Dati/prezzi_zonali_2025_settimana.csv")
prezzi["timestamp"] = pd.to_datetime(prezzi["timestamp"])
mask_nord = prezzi["zona"] == "NORD"
nord = prezzi[mask_nord]
nord.head(3)

Dati, x, y: il grafico a linee più semplice che esista. `fig` è l'oggetto grafico; `fig.show()`
lo disegna. Con il mouse sopra la linea compaiono i valori, trascinando si ingrandisce una zona,
con un doppio clic si torna indietro.

In [ ]:
fig = px.line(nord, x="timestamp", y="eur_mwh")
fig.show()

Dentro `fig` ci sono le tracce, `fig.data`, e il layout, `fig.layout`. Una linea è una traccia
di tipo `scatter` disegnata in modalità linee: in Plotly linee e punti sono la stessa famiglia.

In [ ]:
print(len(fig.data), fig.data[0].type, fig.data[0].mode)

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

In un notebook basta anche scrivere `fig` come ultima riga della cella. `fig.show()` funziona
anche in uno script, quindi è l'abitudine da prendere.

</div>

<a id="sez-2"></a>
## 2. Linee con `px.line`

Sei zone, una linea ciascuna: `color="zona"` fa una traccia per ogni valore distinto della
colonna e aggiunge la legenda. `title` mette il titolo.

In [ ]:
fig = px.line(prezzi, x="timestamp", y="eur_mwh", color="zona", title="Prezzi zonali, 2-8 giugno 2025")
fig.show()

La legenda è viva: un clic su una zona la nasconde, un doppio clic la isola. La SICI sopra a
tutte e il NORD con il buco del 4 giugno si vedono senza calcolare niente.

Le etichette degli assi sono i nomi delle colonne, e `eur_mwh` sull'asse di un grafico per il
capo non si può vedere. `labels` le traduce con un dizionario, senza rinominare il DataFrame.

In [ ]:
etichette = {"timestamp": "Ora", "eur_mwh": "Prezzo [€/MWh]", "zona": "Zona"}

fig = px.line(prezzi, x="timestamp", y="eur_mwh", color="zona", labels=etichette,
              title="Prezzi zonali, 2-8 giugno 2025")
fig.show()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Disegna in `fig_sud` il prezzo della sola zona SUD: titolo `Zona SUD`, asse y etichettato
`Prezzo [€/MWh]`.

</div>

In [ ]:
mask_sud = ...
fig_sud = px.line(...)
fig_sud.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(fig_sud.data) == 1, "❌ Una zona sola: filtra il DataFrame prima di passarlo a px.line"
assert fig_sud.layout.title.text == "Zona SUD", "❌ Il titolo va passato con title='Zona SUD'"
assert fig_sud.layout.yaxis.title.text == "Prezzo [€/MWh]", "❌ L'etichetta dell'asse y si cambia con labels={'eur_mwh': ...}"
print("✅ Tutto corretto")

<a id="sez-3"></a>
## 3. Istogrammi e punti

Un istogramma risponde a "come sono distribuiti i valori": quante ore sotto i 90 €/MWh, quante
sopra i 130. Serve solo la x; l'altezza delle barre la conta Plotly. `nbins` è il numero di
barre, indicativo.

In [ ]:
fig = px.histogram(prezzi, x="eur_mwh", nbins=30, labels=etichette, title="Distribuzione dei prezzi orari")
fig.show()

Con `color` le distribuzioni si separano per zona; `barmode="overlay"` le sovrappone invece di
impilarle e `opacity` le rende trasparenti, così si vede chi sta dove.

In [ ]:
fig = px.histogram(prezzi, x="eur_mwh", color="zona", nbins=30, barmode="overlay", opacity=0.6,
                   labels=etichette, title="Distribuzione dei prezzi per zona")
fig.show()

Lo scatter mette un punto per riga e serve a vedere la relazione tra due numeri. Il caso da
manuale è la turbina: vento sulla x, potenza sulla y. Il timestamp lo costruiamo come nel
notebook sulle date e rinominiamo due colonne, perché `Wind speed | (m/s)` è scomodo da scrivere.

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")
turbina["timestamp"] = pd.to_datetime("2023 " + turbina["Time stamp"], format="%Y %b %d, %I:%M %p")
turbina = turbina.rename(columns={"System power generated | (kW)": "potenza_kw", "Wind speed | (m/s)": "vento_ms"})
turbina.head(3)

In [ ]:
fig = px.scatter(turbina, x="vento_ms", y="potenza_kw", opacity=0.3,
                 labels={"vento_ms": "Vento [m/s]", "potenza_kw": "Potenza [kW]"},
                 title="Curva di potenza della turbina")
fig.show()

La S è la curva di potenza: sotto i 3 m/s la turbina non parte, poi sale, poi satura a 3 MW. Un
punto con vento buono e potenza zero sarebbe un fermo o una misura da controllare: qui non ce ne
sono, e in un grafico lo si vede in un secondo, in una tabella di 8760 righe mai. `opacity` serve
a far vedere dove i punti si accumulano.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Com'è distribuito il vento in Texas? Costruisci in `fig_vento` l'istogramma della colonna
`vento_ms` con 40 barre.

</div>

In [ ]:
fig_vento = px.histogram(...)
fig_vento.show()

In [ ]:
# Verifica: esegui senza modificare
assert fig_vento.data[0].type == "histogram", "❌ Serve px.histogram, non px.line o px.scatter"
assert len(fig_vento.data) == 1, "❌ Una sola distribuzione: niente color"
assert fig_vento.data[0].nbinsx == 40, "❌ Servono 40 barre: nbins=40"
assert max(fig_vento.data[0].x) == turbina["vento_ms"].max(), "❌ Sull'asse x va la colonna vento_ms"
print("✅ Tutto corretto")

<a id="sez-4"></a>
## 4. Più serie e navigazione

I prezzi sono in formato long: una riga per ogni coppia (ora, zona), e `color` separa le serie.
Terna è in formato wide: carico e previsione sono due colonne della stessa riga. Per il wide si
passa a `y` una lista di colonne. Carichiamo il 2024 con la pulizia del notebook sulle date e
teniamo febbraio.

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date").drop_duplicates(subset="Date")

mask_febbraio = carico["Date"].dt.month == 2
febbraio = carico[mask_febbraio]
len(febbraio)

In [ ]:
fig = px.line(febbraio, x="Date", y=["Total Load [MW]", "Forecast Total Load [MW]"],
              title="Carico Nord, febbraio 2024")
fig.show()

Con la `y` a lista Plotly chiama la serie `variable` e il valore `value`: sono i nomi da mettere
in `labels`. Il layout si ritocca dopo con `fig.update_layout`: altezza, titolo della legenda,
etichetta dell'asse y.

In [ ]:
fig = px.line(febbraio, x="Date", y=["Total Load [MW]", "Forecast Total Load [MW]"],
              labels={"Date": "Data", "value": "MW", "variable": "Serie"}, title="Carico Nord, febbraio 2024")
fig.update_layout(height=450, legend_title_text="", yaxis_title="Carico [MW]")
fig.show()

<div style="background:rgba(126,87,194,0.12); border-left:4px solid #7e57c2; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📘 Approfondimento · Da wide a long con melt (puoi saltarlo)**

L'altra strada è portare il DataFrame in formato long con `melt`, e poi usare `color` come con i
prezzi. Utile quando le colonne da confrontare sono tante o cambiano:

```python
lungo = febbraio.melt(id_vars="Date", value_vars=["Total Load [MW]", "Forecast Total Load [MW]"],
                      var_name="serie", value_name="mw")
px.line(lungo, x="Date", y="mw", color="serie")
```

</div>

Un mese di quarti d'ora sono 2784 punti: troppi per leggerli tutti insieme. Lo slider sotto il
grafico e i bottoni del `rangeselector` servono a navigarli. I bottoni si descrivono con una
lista di dizionari: `count` e `step` dicono quanto indietro, `label` cosa c'è scritto.

In [ ]:
bottoni = [
    dict(count=1, step="day", stepmode="backward", label="1 giorno"),
    dict(count=7, step="day", stepmode="backward", label="1 settimana"),
    dict(step="all", label="tutto"),
]
fig.update_xaxes(rangeslider_visible=True, rangeselector=dict(buttons=bottoni))
fig.show()

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

Il `rangeselector` nessuno lo sa a memoria: si copia da qui o dalla documentazione. Quello che va
ricordato è il nome, per sapere cosa cercare.

</div>

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Costruisci in `fig_marzo` il grafico del solo `Total Load [MW]` di marzo 2024, con lo slider
sotto l'asse x.

</div>

In [ ]:
mask_marzo = ...
fig_marzo = px.line(...)
fig_marzo.update_xaxes(...)
fig_marzo.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(fig_marzo.data) == 1, "❌ Una serie sola: y='Total Load [MW]' come stringa, non come lista"
assert fig_marzo.layout.xaxis.rangeslider.visible is True, "❌ Lo slider si accende con fig.update_xaxes(rangeslider_visible=True)"
print("✅ Tutto corretto")

<a id="sez-5"></a>
## 5. Esportare

`fig.write_html` salva il grafico in un file che si apre con qualunque browser, interattività
compresa: chi lo riceve non ha bisogno di Python. Il file finisce nella cartella del notebook.

In [ ]:
from pathlib import Path

fig.write_html("carico_febbraio.html")
print(f"{Path('carico_febbraio.html').stat().st_size / 1_000_000:.1f} MB")

Cinque MB per un grafico: dentro c'è tutta la libreria Plotly, così il file funziona anche senza
rete. Con `include_plotlyjs="cdn"` la libreria viene scaricata all'apertura e il file scende a
poche centinaia di KB, ma chi lo apre deve essere connesso.

In [ ]:
fig.write_html("carico_febbraio_leggero.html", include_plotlyjs="cdn")
print(f"{Path('carico_febbraio_leggero.html').stat().st_size / 1_000:.0f} KB")

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

Per un'immagine fissa c'è `fig.write_image("grafico.png")`, che però vuole una libreria in più
(`kaleido`, con `uv add`). Qui basta l'HTML: nessuno deve installare niente per aprirlo.

</div>

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- `px.line`, `px.histogram`, `px.scatter`: stessi argomenti, `(df, x=, y=, color=, labels=, title=)`, e restituiscono `fig`.
- `fig.update_layout(...)` e `fig.update_xaxes(...)` ritoccano dopo; `fig.show()` disegna.
- `fig.write_html("nome.html")` per chi non ha Python.

</div>

<a id="sez-6"></a>
## 6. Esercizi

<a id="es-10-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 10.1 · Il grafico per il capo**

Il capo vuole entro stasera il grafico del carico del Nord di febbraio 2024, da aprire sul suo
portatile in riunione, e sul suo portatile non c'è Python. Vuole una linea sola, poter
zoomare su un giorno, e un titolo che si capisca.

**Cosa fare.**

1. Da `febbraio` costruisci `fig_capo`: linea del solo `Total Load [MW]` sulla `Date`, titolo
   `Carico Nord, febbraio 2024`, asse y etichettato `Carico [MW]`.
2. Accendi lo slider sotto l'asse x.
3. Salvalo in `carico_febbraio_capo.html`, nella cartella del notebook.

*Suggerimento:* L'etichetta dell'asse si cambia con `labels={"Total Load [MW]": "Carico [MW]"}` oppure dopo, con `update_layout(yaxis_title=...)`.

</div>

In [ ]:
fig_capo = px.line(...)
fig_capo.update_xaxes(...)
fig_capo.write_html(...)
fig_capo.show()

In [ ]:
# Verifica: esegui senza modificare
from pathlib import Path
assert len(fig_capo.data) == 1, "❌ Una linea sola: y='Total Load [MW]', senza la previsione"
assert fig_capo.layout.title.text == "Carico Nord, febbraio 2024", "❌ Il titolo deve essere esattamente 'Carico Nord, febbraio 2024'"
assert fig_capo.layout.yaxis.title.text == "Carico [MW]", "❌ L'asse y deve dire 'Carico [MW]': labels oppure update_layout(yaxis_title=...)"
assert fig_capo.layout.xaxis.rangeslider.visible is True, "❌ Manca lo slider: update_xaxes(rangeslider_visible=True)"
assert Path("carico_febbraio_capo.html").exists(), "❌ Il file carico_febbraio_capo.html non c'è: write_html con quel nome esatto"
print("✅ Esercizio 10.1 completato")

<a id="es-10-1-bis"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 10.1 bis · Zone a confronto**

*In alternativa al 10.1: stesso obiettivo, scenario diverso.*

Sara, del trading, vuole vedere NORD e SICI sullo stesso grafico, e solo quelle due:
con sei linee non ci capisce niente. Il file HTML lo allega alla mail del lunedì.

**Cosa fare.**

1. Filtra `prezzi` sulle zone NORD e SICI in `due_zone`.
2. Costruisci `fig_zone`: una linea per zona, titolo `NORD e SICI a confronto`, asse y `Prezzo [€/MWh]`.
3. Salvalo in `zone_confronto.html`.

*Suggerimento:* `isin(["NORD", "SICI"])` per il filtro, `color="zona"` per le due linee.

</div>

In [ ]:
mask_zone = ...
due_zone = prezzi[mask_zone]

fig_zone = px.line(...)
fig_zone.write_html(...)
fig_zone.show()

In [ ]:
# Verifica: esegui senza modificare
from pathlib import Path
assert len(fig_zone.data) == 2, "❌ Due linee, non sei: filtra le zone prima di passare il DataFrame a px.line"
assert {traccia.name for traccia in fig_zone.data} == {"NORD", "SICI"}, "❌ Le due tracce devono essere NORD e SICI: color='zona' sul DataFrame filtrato"
assert fig_zone.layout.title.text == "NORD e SICI a confronto", "❌ Controlla il titolo, deve essere esatto"
assert Path("zone_confronto.html").exists(), "❌ Manca il file zone_confronto.html"
print("✅ Esercizio 10.1 bis completato")

<a id="es-10-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 10.2 · Vento e potenza**

Elena, che segue l'eolico, deve presentare la turbina texana al comitato investimenti:
vuole la curva di potenza (vento contro potenza) e la distribuzione della potenza prodotta,
per far vedere quante ore la macchina lavora a pieno regime e quante sta ferma.

**Cosa fare.**

1. `fig_curva`: scatter di `turbina` con il vento sulla x e la potenza sulla y, con `opacity=0.3`
   e titolo `Curva di potenza`.
2. `fig_dist`: istogramma della colonna `potenza_kw`, 30 barre, titolo `Distribuzione della potenza`.

*Suggerimento:* Le colonne si chiamano `vento_ms` e `potenza_kw`: le abbiamo rinominate noi.

</div>

In [ ]:
fig_curva = px.scatter(...)
fig_curva.show()

fig_dist = px.histogram(...)
fig_dist.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(fig_curva.data) == 1 and "scatter" in fig_curva.data[0].type, "❌ fig_curva: px.scatter con una sola traccia (niente color)"
assert fig_curva.data[0].marker.opacity == 0.3, "❌ fig_curva: passa opacity=0.3 a px.scatter"
assert fig_dist.data[0].type == "histogram", "❌ fig_dist: serve px.histogram"
assert fig_dist.layout.title.text == "Distribuzione della potenza", "❌ fig_dist: controlla il titolo"
print("✅ Esercizio 10.2 completato")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Un passo in più (facoltativo)**

Il vento cambia con le stagioni? Crea in `turbina` una colonna `mese` con il nome del
mese (`turbina["timestamp"].dt.month_name()`, come `day_name()` dava il giorno) e rifai lo
scatter in `fig_mesi` con `color="mese"`: una traccia per mese, e la legenda per accenderli
uno alla volta.

</div>

In [ ]:
turbina["mese"] = ...
fig_mesi = px.scatter(...)
fig_mesi.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(fig_mesi.data) == 12, "❌ fig_mesi: una traccia per mese, dodici in tutto; il colore deve venire dal nome del mese"
assert "January" in {traccia.name for traccia in fig_mesi.data}, "❌ fig_mesi: la colonna mese deve contenere il nome del mese (dt.month_name())"
print("✅ Esercizio 10.2 · un passo in più completato")

<a id="es-10-2-bis"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 10.2 bis · Temperatura e potenza**

*In alternativa al 10.2: stesso obiettivo, scenario diverso.*

Chi gestisce la manutenzione sospetta che col caldo la turbina renda meno. Vuole vedere la
temperatura dell'aria contro la potenza prodotta, e come sono distribuite le temperature
nell'anno, prima di chiedere un controllo al costruttore.

**Cosa fare.**

1. Rinomina in `turbina` la colonna `Air temperature | ('C)` in `temperatura_c`.
2. `fig_temp`: scatter con la temperatura sulla x e la potenza sulla y, con `opacity=0.3`
   e titolo `Temperatura e potenza`.
3. `fig_temp_dist`: istogramma della colonna `temperatura_c`, 30 barre, titolo
   `Distribuzione della temperatura`.

*Suggerimento:* Il nome della colonna da cambiare si copia dall'intestazione del file: `rename(columns={...})` come con il vento.

</div>

In [ ]:
turbina = turbina.rename(columns=...)

fig_temp = px.scatter(...)
fig_temp.show()

fig_temp_dist = px.histogram(...)
fig_temp_dist.show()

In [ ]:
# Verifica: esegui senza modificare
assert "temperatura_c" in turbina.columns, "❌ La colonna va rinominata in temperatura_c con rename(columns={...})"
assert len(fig_temp.data) == 1 and "scatter" in fig_temp.data[0].type, "❌ fig_temp: px.scatter con una sola traccia (niente color)"
assert fig_temp.data[0].marker.opacity == 0.3, "❌ fig_temp: passa opacity=0.3 a px.scatter"
assert max(fig_temp.data[0].x) == turbina["temperatura_c"].max(), "❌ fig_temp: la temperatura va sull'asse x"
assert fig_temp_dist.data[0].type == "histogram", "❌ fig_temp_dist: serve px.histogram"
assert fig_temp_dist.data[0].nbinsx == 30, "❌ fig_temp_dist: servono 30 barre, nbins=30"
assert max(fig_temp_dist.data[0].x) == turbina["temperatura_c"].max(), "❌ fig_temp_dist: sull'asse x va la colonna temperatura_c"
assert fig_temp_dist.layout.title.text == "Distribuzione della temperatura", "❌ fig_temp_dist: controlla il titolo"
print("✅ Esercizio 10.2 bis completato")

---

**Fine del notebook 10.** Prossimo: [11 · Agenti per il coding](11_Agenti_per_il_coding.ipynb)